# Passo 5 – Análises de robustez

O Passo 4 trouxe dois alertas. Este notebook testa os dois.

**Alerta 1 – atalhos.** As palavras mais pesadas do Fake.Br incluem nomes de veículos e marcas de formatação (por exemplo "rádio eldorado", "degravação", "ouvir clique"). O modelo pode estar reconhecendo **de onde veio o texto**, e não a desinformação.

- **Teste A:** tirar as palavras mais pesadas, aos poucos (10, 25, 50... até 1.000), e ver se o desempenho cai rápido.
- **Teste B:** tirar uma lista de marcas de veículo (lista exploratória, tirada das palavras pesadas do treino).

**Alerta 2 – tamanho.** O Fake.Br só tem textos longos (mediana de 158 palavras). Os boatos eleitorais são curtos (mediana de 38).

- **Teste C:** desempenho por faixa de tamanho no teste do Fake.Br.
- **Teste D:** treinar com os textos **cortados nas primeiras 40 palavras** e repetir o teste externo.
- **Teste E:** medir, em textos curtos da FakeRecogna, a taxa de acerto nos falsos **e** a taxa de falsos positivos nos verdadeiros.

**Como ler:** se o resultado se mantém depois dos cortes, a conclusão fica mais firme. Se cai muito, a conclusão precisa de ressalva.

**Pré-requisito:** notebooks 00 a 04 executados (precisa do `melhores_parametros.json`).

In [ ]:
import os, re, unicodedata, json, ast, warnings
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, confusion_matrix
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')
PROC = os.path.join(PASTA_TCC, 'dados', 'processados')
RES = os.path.join(PASTA_TCC, 'resultados')
SEED = 42
np.random.seed(SEED)

fake = pd.read_csv(os.path.join(PROC, 'fakebr_prep.csv'))
reco = pd.read_csv(os.path.join(PROC, 'recogna_prep.csv'))
fpc = pd.read_csv(os.path.join(PROC, 'fpc_prep.csv'))
for d in (fake, reco, fpc):
    d['texto_proc'] = d['texto_proc'].fillna('')
for d in (fake, reco):
    d['rotulo'] = d['rotulo'].astype(int)
tr = fake[fake.particao == 'treino'].reset_index(drop=True)
te = fake[fake.particao == 'teste'].reset_index(drop=True)
reco_tr = reco[reco.particao == 'treino'].reset_index(drop=True)
reco_te = reco[reco.particao == 'teste'].reset_index(drop=True)

# Parâmetros escolhidos no Passo 4 (versão sem stemming)
with open(os.path.join(RES, 'melhores_parametros.json')) as f:
    PARAMS = {k.split('|')[0]: {p: ast.literal_eval(v) for p, v in d.items()}
              for k, d in json.load(f).items() if k.endswith('sem_stem')}
print({m: p for m, p in PARAMS.items() if m != 'XGBoost'})

In [ ]:
import nltk
# O Colab apaga estes recursos quando a sessão acaba, então baixamos de novo (rápido)
for recurso in ['stopwords', 'rslp']:
    nltk.download(recurso, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

NEGACOES = {'não', 'nem', 'nunca', 'jamais', 'nenhum', 'nenhuma', 'sem', 'nada', 'ninguém'}
STOP = set(stopwords.words('portuguese')) - NEGACOES
STEM = RSLPStemmer()

RE_URL = re.compile(r'(https?://\S+|www\.\S+)', re.I)
RE_MENCAO = re.compile(r'@\w+')
RE_HASHTAG = re.compile(r'#\w+')
RE_MARCA = re.compile(r'\(sic\.?\)|\[sic\.?\]|\(\.\.\.\)|\[\.\.\.\]', re.I)
RE_NUM = re.compile(r'\d+([.,]\d+)*')
RE_NAOLETRA = re.compile(r'[^a-zA-ZÀ-ÿ ]')

def normalizar(texto):
    t = str(texto)
    t = unicodedata.normalize('NFC', t)
    t = RE_URL.sub(' ', t)
    t = RE_MARCA.sub(' ', t)
    t = RE_MENCAO.sub(' ', t)
    t = RE_HASHTAG.sub(' ', t)
    t = t.lower()
    t = RE_NUM.sub(' num ', t)
    t = RE_NAOLETRA.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenizar(texto):
    return [p for p in normalizar(texto).split() if len(p) > 1 or p == 'e']

def sem_stopwords(tokens):
    return [t for t in tokens if t not in STOP]

def preparar(texto):
    toks = sem_stopwords(tokenizar(texto))
    return ' '.join(toks), ' '.join(STEM.stem(t) for t in toks)

def preparar_proc(texto):
    return ' '.join(sem_stopwords(tokenizar(texto)))

In [ ]:
def novo_modelo(nome, stop_words=None):
    tf = TfidfVectorizer(min_df=2, max_features=50000, sublinear_tf=True, stop_words=list(stop_words) if stop_words else None)
    clf = MultinomialNB() if nome == 'Naive Bayes' else LinearSVC(random_state=SEED, max_iter=5000)
    p = Pipeline([('tfidf', tf), ('clf', clf)])
    p.set_params(**PARAMS[nome])
    return p

def taxas(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {'f1_falso': round(f1_score(y_true, y_pred, zero_division=0), 4),
            'recall_falso': round(tp / (tp + fn), 4) if tp + fn else np.nan,
            'taxa_falso_positivo': round(fp / (fp + tn), 4) if fp + tn else np.nan, 'n': len(y_true)}

MODELOS = ['SVM', 'Naive Bayes']

## Teste A – Tirar as palavras mais pesadas, aos poucos

As palavras são escolhidas **só com o treino** (peso do modelo já treinado). Depois o modelo é treinado de novo sem elas.

- Se o F1 cai pouco: o modelo usa **muitas** palavras. Sinal bom.
- Se cai muito logo no começo: o modelo depende de **poucas** palavras. Sinal de atalho.

In [ ]:
def palavras_pesadas(nome, k):
    m = novo_modelo(nome).fit(tr['texto_proc'], tr['rotulo'])
    nomes = np.array(m.named_steps['tfidf'].get_feature_names_out())
    clf = m.named_steps['clf']
    peso = np.abs(clf.coef_[0]) if nome == 'SVM' else np.abs(clf.feature_log_prob_[1] - clf.feature_log_prob_[0])
    top = nomes[np.argsort(peso)[::-1][:k]]
    return {t for feat in top for t in feat.split()}

linhas = []
for nome in MODELOS:
    for k in [0, 10, 25, 50, 100, 250, 500, 1000]:
        retirar = palavras_pesadas(nome, k) if k else set()
        m = novo_modelo(nome, retirar).fit(tr['texto_proc'], tr['rotulo'])
        r_te = taxas(te['rotulo'], m.predict(te['texto_proc']))
        r_fpc = m.predict(fpc['texto_proc']).mean()
        linhas.append({'modelo': nome, 'k_atributos_retirados': k, 'palavras_retiradas': len(retirar),
                       'f1_teste': r_te['f1_falso'], 'taxa_falso_positivo': r_te['taxa_falso_positivo'],
                       'recall_boatos_eleitorais': round(r_fpc, 4)})
quadro_ablacao = pd.DataFrame(linhas)
quadro_ablacao

## Teste B – Tirar marcas de veículo (lista exploratória)

A lista abaixo foi montada olhando as palavras pesadas **do treino** (Passo 4). Ela é **exploratória e subjetiva**: deve ser declarada como tal. Não é uma lista completa de veículos.

In [ ]:
MARCAS = {'eldorado', 'rádio', 'fm', 'degravação', 'ouvir', 'clique', 'play', 'bbc', 'globo', 'folha', 'diário', 'estadão',
          'news', 'sp', 'uol', 'jornal', 'íntegra', 'comentário'}
linhas = []
for nome in MODELOS:
    for rotulo_cond, retirar in [('com todas as palavras', set()), ('sem marcas de veículo', MARCAS)]:
        m = novo_modelo(nome, retirar).fit(tr['texto_proc'], tr['rotulo'])
        r_te = taxas(te['rotulo'], m.predict(te['texto_proc']))
        linhas.append({'modelo': nome, 'condicao': rotulo_cond, 'f1_teste': r_te['f1_falso'],
                       'taxa_falso_positivo': r_te['taxa_falso_positivo'],
                       'recall_boatos_eleitorais': round(m.predict(fpc['texto_proc']).mean(), 4)})
quadro_marcas = pd.DataFrame(linhas)
quadro_marcas

## Teste C – Desempenho por faixa de tamanho (teste do Fake.Br)

Divide o teste em 4 faixas de tamanho (quartis do número de palavras). Se o modelo vai bem só nos textos longos, não dá para esperar o mesmo em boatos curtos.

In [ ]:
faixas = pd.qcut(te['n_palavras_orig'], 4, duplicates='drop')
linhas = []
for nome in MODELOS:
    m = novo_modelo(nome).fit(tr['texto_proc'], tr['rotulo'])
    te['_pred'] = m.predict(te['texto_proc'])
    for faixa, g in te.groupby(faixas):
        linhas.append({'modelo': nome, 'faixa_palavras': str(faixa), **taxas(g['rotulo'], g['_pred'])})
quadro_faixas = pd.DataFrame(linhas)
quadro_faixas

## Teste D – Treinar com textos cortados (primeiras 40 palavras)

Simula o formato curto do boato. Corta **treino, teste interno e boatos** nas primeiras 40 palavras antes do pré-processamento. A mediana do boato eleitoral é 38 palavras.

Aqui o modelo aprende em textos curtos. Compare com o modelo de texto completo.

In [ ]:
K = 40
def cortar(serie, k=K):
    return serie.fillna('').str.split().str[:k].str.join(' ')

tr['txt_curto'] = [preparar_proc(t) for t in cortar(tr['texto'])]
te['txt_curto'] = [preparar_proc(t) for t in cortar(te['texto'])]
fpc['txt_curto'] = [preparar_proc(t) for t in cortar(fpc['texto'])]
reco_te['txt_curto'] = [preparar_proc(t) for t in cortar(reco_te['texto'])]

linhas = []
for nome in MODELOS:
    for cond, col_tr, col_te, col_fpc in [('texto completo', 'texto_proc', 'texto_proc', 'texto_proc'),
                                           (f'cortado em {K} palavras', 'txt_curto', 'txt_curto', 'txt_curto')]:
        m = novo_modelo(nome).fit(tr[col_tr], tr['rotulo'])
        r = taxas(te['rotulo'], m.predict(te[col_te]))
        linhas.append({'modelo': nome, 'treino_e_teste': cond, 'f1_teste_fakebr': r['f1_falso'],
                       'taxa_falso_positivo_fakebr': r['taxa_falso_positivo'],
                       'recall_boatos_eleitorais': round(m.predict(fpc[col_fpc]).mean(), 4)})
quadro_corte = pd.DataFrame(linhas)
quadro_corte

## Teste E – Textos curtos da FakeRecogna (modelo treinado no Fake.Br)

A FakeRecogna tem notícias verdadeiras e falsas **curtas**. Aplicamos nelas o modelo treinado no Fake.Br e medimos, só nos textos de até 60 palavras:

- **recall nos falsos** (quantos falsos curtos o modelo pega);
- **taxa de falsos positivos** (quantos verdadeiros curtos ele marca como falso).

Se o recall é alto mas os falsos positivos também, o modelo só marca como falso o que é curto.

In [ ]:
curtos = reco_te[reco_te['n_palavras_orig'] <= 60].reset_index(drop=True)
longos = reco_te[reco_te['n_palavras_orig'] > 60].reset_index(drop=True)
print('FakeRecogna teste: curtos', len(curtos), '| longos', len(longos))
linhas = []
for nome in MODELOS:
    m = novo_modelo(nome).fit(tr['texto_proc'], tr['rotulo'])
    for rotulo_faixa, g in [('até 60 palavras', curtos), ('mais de 60 palavras', longos)]:
        linhas.append({'modelo': nome, 'faixa': rotulo_faixa, **taxas(g['rotulo'], m.predict(g['texto_proc']))})
quadro_curtos = pd.DataFrame(linhas)
quadro_curtos

## Salvando

In [ ]:
quadro_ablacao.to_csv(os.path.join(RES, 'quadro8a_ablacao_palavras.csv'), index=False)
quadro_marcas.to_csv(os.path.join(RES, 'quadro8b_sem_marcas_veiculo.csv'), index=False)
quadro_faixas.to_csv(os.path.join(RES, 'quadro9a_faixas_tamanho.csv'), index=False)
quadro_corte.to_csv(os.path.join(RES, 'quadro9b_texto_cortado.csv'), index=False)
quadro_curtos.to_csv(os.path.join(RES, 'quadro9c_fakerecogna_curtos.csv'), index=False)
print('Resultados salvos em', RES)